# Step 3｜时间切分与Final OOT锁定

## 目的

按照时间先后顺序划分Train、Validation、Calibration和Final OOT，
并将最终边界冻结在`configs/split_v1.yaml`中。

## 切分原则

- 使用`WEEK_NUM`作为主要切分轴；
- 不进行随机划分；
- Final OOT必须来自最新的有标签时间段；
- 切分边界依据周度样本数和事件数确定；
- 边界冻结后，不因模型结果不理想而修改；
- Final OOT只用于最终评估，不参与训练、调参、校准或阈值选择。

## 本阶段不做

- 不读取其他特征表；
- 不构造模型特征；
- 不训练模型；
- 不查看任何Final OOT模型指标。

In [2]:
# Step 3.1：读取时间切分所需字段
from pathlib import Path
import pandas as pd

DATA_ROOT = Path(r"D:\Home_Credit_datasets")
PROJECT_ROOT = Path(r"D:\Home_Credit_datasets")

TRAIN_BASE_PATH = (
    DATA_ROOT
    / "parquet_files"
    / "train"
    / "train_base.parquet"
)

CONFIG_DIR = PROJECT_ROOT / "configs"
SPLIT_CONFIG_PATH = CONFIG_DIR / "split_v1.yaml"

CONFIG_DIR.mkdir(
    parents = True,
    exist_ok = True
)

base = pd.read_parquet(
    TRAIN_BASE_PATH,
    columns=[
        "case_id",
        "date_decision",
        "WEEK_NUM",
        "target"
    ]
)

base["date_decision"] = pd.to_datetime(
    base["date_decision"]
)

weekly_summary = (
    base.groupby("WEEK_NUM", as_index=False)
    .agg(
        start_date=("date_decision", "min"),
        end_date=("date_decision", "max"),
        sample_count=("case_id", "size"),
        event_count=("target", "sum"),
        event_rate=("target", "mean")
    )
)

print("总样本数：", len(base))
print("周数：", len(weekly_summary))
print(
    "WEEK_NUM范围：",
    weekly_summary["WEEK_NUM"].min(),
    "至",
    weekly_summary["WEEK_NUM"].max()
)
print("配置文件：", SPLIT_CONFIG_PATH)

display(weekly_summary.head())
display(weekly_summary.tail())

总样本数： 1526659
周数： 92
WEEK_NUM范围： 0 至 91
配置文件： D:\Home_Credit_datasets\configs\split_v1.yaml


,WEEK_NUM,start_date,end_date,sample_count,event_count,event_rate
0,0,2019-01-01,2019-01-07,16735,400,0.023902
1,1,2019-01-08,2019-01-14,18841,472,0.025052
2,2,2019-01-15,2019-01-21,17476,465,0.026608
3,3,2019-01-22,2019-01-28,16108,419,0.026012
4,4,2019-01-29,2019-02-04,14309,404,0.028234


,WEEK_NUM,start_date,end_date,sample_count,event_count,event_rate
87,87,2020-09-01,2020-09-07,17886,329,0.018394
88,88,2020-09-08,2020-09-14,14234,265,0.018617
89,89,2020-09-15,2020-09-21,13600,309,0.022721
90,90,2020-09-22,2020-09-28,12103,330,0.027266
91,91,2020-09-29,2020-10-05,12674,288,0.022724


In [8]:
# Step 3.2：比较候选时间切分方案
candidate_splits = {
    "A_12周": {
        "Model Train": (0, 55),
        "Tuning Validation": (56, 67),
        "Calibration/Policy Validation": (68, 79),
        "Final OOT": (80, 91)
    },
    "B_10周": {
        "Model Train": (0, 61),
        "Tuning Validation": (62, 71),
        "Calibration/Policy Validation": (72, 81),
        "Final OOT": (82, 91)
    },
    "C_8周": {
        "Model Train": (0, 67),
        "Tuning Validation": (68, 75),
        "Calibration/Policy Validation": (76, 83),
        "Final OOT": (84, 91)
    }
}

split_rows = []

for candidate_name, split_ranges in candidate_splits.items():
    for split_name, (start_week, end_week) in split_ranges.items():

        segment = weekly_summary[
            weekly_summary["WEEK_NUM"].between(
                start_week,
                end_week
            )
        ]

        sample_count = segment["sample_count"].sum()
        event_count = segment["event_count"].sum()

        split_rows.append({
            "candidate": candidate_name,
            "split": split_name,
            "start_week": start_week,
            "end_week": end_week,
            "start_date": segment["start_date"].min(),
            "end_date": segment["end_date"].max(),
            "week_count": len(segment),
            "sample_count": sample_count,
            "event_count": event_count,
            "event_rate": event_count / sample_count
        })

candidate_summary = pd.DataFrame(split_rows)

display(candidate_summary)

,candidate,split,start_week,end_week,start_date,end_date,week_count,sample_count,event_count,event_rate
0,A_12周,Model Train,0,55,2019-01-01,2020-01-27,56,1148308,36099,0.031437
1,A_12周,Tuning Validation,56,67,2020-01-28,2020-04-20,12,152012,6925,0.045556
2,A_12周,Calibration/Policy Validation,68,79,2020-04-21,2020-07-13,12,88545,2062,0.023288
3,A_12周,Final OOT,80,91,2020-07-14,2020-10-05,12,137794,2908,0.021104
4,B_10周,Model Train,0,61,2019-01-01,2020-03-09,62,1250581,41040,0.032817
5,B_10周,Tuning Validation,62,71,2020-03-10,2020-05-18,10,63875,2298,0.035977
6,B_10周,Calibration/Policy Validation,72,81,2020-05-19,2020-07-27,10,87398,2056,0.023525
7,B_10周,Final OOT,82,91,2020-07-28,2020-10-05,10,124805,2600,0.020832
8,C_8周,Model Train,0,67,2019-01-01,2020-04-20,68,1300320,43024,0.033087
9,C_8周,Tuning Validation,68,75,2020-04-21,2020-06-15,8,56617,1303,0.023014


Step 3.2完成。建议选择 B_10周方案：

|数据段 | 周范围 | 样本数 | 事件数 | 事件率
| :--: | :--: | :--: | :--: | :--: |
|Model Train | 0–61 | 1,250,581 | 41,040 | 3.2817%
Tuning Validation | 62–71 | 63,875 | 2,298 | 3.5977%
Calibration/Policy Validation | 72–81 | 87,398 | 2,056 | 2.3525%
Final OOT | 82–91 | 124,805 | 2,600 | 2.0832%

选择原因：

- 第63—70周的主要阶段变化完整落在Validation中，可以检验模型面对时间变化时是否稳定；
- Calibration从第72周开始，避开主要变化期，适合校准概率和选择策略；
- Final OOT保留最新10周，共12.48万条样本、2,600个事件，支持充足；
- A方案把变化期拆到Validation和Calibration两段；
- C方案让训练集吸收了部分变化期，而且三个验证阶段只有8周。

In [9]:
# Step 3.3：验证B方案没有遗漏或重叠
selected_ranges = candidate_splits["B_10周"]

split_order = list(selected_ranges.keys())

split_assignment = pd.Series(
    pd.NA,
    index=base.index,
    dtype="string"
)

for split_name, (start_week, end_week) in selected_ranges.items():

    mask = base["WEEK_NUM"].between(
        start_week,
        end_week
    )

    if split_assignment.loc[mask].notna().any():
        raise ValueError(
            f"{split_name}与其他数据段存在重叠"
        )

    split_assignment.loc[mask] = split_name

unassigned_count = split_assignment.isna().sum()

if unassigned_count != 0:
    raise ValueError(
        f"存在{unassigned_count}条未分配样本"
    )


split_audit_base = base.assign(
    split=split_assignment
)

split_audit_base["split"] = pd.Categorical(
    split_audit_base["split"],
    categories=split_order,
    ordered=True
)

selected_summary = (
    split_audit_base.groupby(
        "split",
        as_index=False,
        observed=True
    )
    .agg(
        start_week=("WEEK_NUM", "min"),
        end_week=("WEEK_NUM", "max"),
        start_date=("date_decision", "min"),
        end_date=("date_decision", "max"),
        week_count=("WEEK_NUM", "nunique"),
        sample_count=("case_id", "size"),
        event_count=("target", "sum"),
        event_rate=("target", "mean")
    )
)


selected_summary["non_event_count"] = (
    selected_summary["sample_count"]
    - selected_summary["event_count"]
)


print("未分配样本数：", unassigned_count)
print(
    "各段样本数合计：",
    selected_summary["sample_count"].sum()
)
print(
    "是否等于总样本数：",
    selected_summary["sample_count"].sum()
    == len(base)
)

display(selected_summary)

未分配样本数： 0
各段样本数合计： 1526659
是否等于总样本数： True


,split,start_week,end_week,start_date,end_date,week_count,sample_count,event_count,event_rate,non_event_count
0,Model Train,0,61,2019-01-01,2020-03-09,62,1250581,41040,0.032817,1209541
1,Tuning Validation,62,71,2020-03-10,2020-05-18,10,63875,2298,0.035977,61577
2,Calibration/Policy Validation,72,81,2020-05-19,2020-07-27,10,87398,2056,0.023525,85342
3,Final OOT,82,91,2020-07-28,2020-10-05,10,124805,2600,0.020832,122205


In [4]:
# Step 3.4：写入并冻结split_v1.yaml
PROJECT_ROOT = Path(r"D:\Risk_control project")

CONFIG_DIR = PROJECT_ROOT / "configs"
SPLIT_CONFIG_PATH = CONFIG_DIR / "split_v1.yaml"

CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

split_config_text = """
split_id: split_v1
status: frozen

dataset: "Home Credit - Credit Risk Model Stability 2024"
source_table: train_base.parquet

time_column: date_decision
week_column: WEEK_NUM
target_column: target
split_method: chronological_by_week
selected_candidate: B_10_week

splits:
  model_train:
    display_name: "Model Train"
    start_week: 0
    end_week: 61
    start_date: "2019-01-01"
    end_date: "2020-03-09"
    sample_count: 1250581
    event_count: 41040
    non_event_count: 1209541
    event_rate: 0.032817

  tuning_validation:
    display_name: "Tuning Validation"
    start_week: 62
    end_week: 71
    start_date: "2020-03-10"
    end_date: "2020-05-18"
    sample_count: 63875
    event_count: 2298
    non_event_count: 61577
    event_rate: 0.035977

  calibration_policy_validation:
    display_name: "Calibration/Policy Validation"
    start_week: 72
    end_week: 81
    start_date: "2020-05-19"
    end_date: "2020-07-27"
    sample_count: 87398
    event_count: 2056
    non_event_count: 85342
    event_rate: 0.023525

  final_oot:
    display_name: "Final OOT"
    start_week: 82
    end_week: 91
    start_date: "2020-07-28"
    end_date: "2020-10-05"
    sample_count: 124805
    event_count: 2600
    non_event_count: 122205
    event_rate: 0.020832

selection_reason:
  - "Maintain chronological order without random splitting."
  - "Keep the main week 63-70 regime change inside Tuning Validation."
  - "Reserve weeks 72-81 for calibration and policy development."
  - "Reserve the latest 10 labeled weeks as Final OOT."

final_oot_rules:
  locked: true
  allowed_use: final_evaluation_only
  forbidden_uses:
    - model_training
    - feature_selection
    - hyperparameter_tuning
    - probability_calibration
    - threshold_selection
""".strip() + "\n"

if SPLIT_CONFIG_PATH.exists():

    existing_text = SPLIT_CONFIG_PATH.read_text(
        encoding="utf-8"
    )

    if existing_text != split_config_text:
        raise FileExistsError(
            "split_v1.yaml已存在且内容不同，"
            "禁止直接覆盖已冻结的切分配置。"
        )

    print("配置文件已存在，且内容一致。")

else:

    SPLIT_CONFIG_PATH.write_text(
        split_config_text,
        encoding="utf-8"
    )

    print("配置文件创建成功。")


print("配置文件：", SPLIT_CONFIG_PATH)
print("文件是否存在：", SPLIT_CONFIG_PATH.exists())

配置文件已存在，且内容一致。
配置文件： D:\Risk_control project\configs\split_v1.yaml
文件是否存在： True


In [5]:
# Step 3.5：建立Final OOT访问日志
METADATA_DIR = PROJECT_ROOT / "metadata"
OOT_ACCESS_LOG_PATH = (
    METADATA_DIR
    / "oot_access_log.csv"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

access_record = {
    "access_id": "OOT-BOUNDARY-001",
    "access_time": pd.Timestamp.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
    "project_stage": "Step 3",
    "split_id": "split_v1",
    "oot_week_range": "82-91",
    "purpose": "Define and freeze temporal split",
    "data_scope": "Aggregated sample and event support only",
    "fields_accessed": (
        "case_id|date_decision|WEEK_NUM|target"
    ),
    "feature_data_accessed": False,
    "model_predictions_accessed": False,
    "model_metrics_accessed": False,
    "used_for_model_selection": False,
    "final_oot_model_evaluation_opened": False,
    "notes": (
        "Only sample_count, event_count and event_rate "
        "were inspected for split design. "
        "Final OOT model evaluation remains sealed."
    )
}

if OOT_ACCESS_LOG_PATH.exists():

    oot_access_log = pd.read_csv(
        OOT_ACCESS_LOG_PATH
    )

    if "access_id" not in oot_access_log.columns:
        raise ValueError(
            "现有OOT访问日志缺少access_id字段"
        )

    if access_record["access_id"] not in (
        oot_access_log["access_id"].tolist()
    ):
        oot_access_log = pd.concat(
            [
                oot_access_log,
                pd.DataFrame([access_record])
            ],
            ignore_index=True
        )

        oot_access_log.to_csv(
            OOT_ACCESS_LOG_PATH,
            index=False,
            encoding="utf-8-sig"
        )

        print("OOT访问记录已追加。")

    else:
        print("该访问记录已经存在，不重复写入。")

else:

    oot_access_log = pd.DataFrame(
        [access_record]
    )

    oot_access_log.to_csv(
        OOT_ACCESS_LOG_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("OOT访问日志创建成功。")


print("访问日志：", OOT_ACCESS_LOG_PATH)
print(
    "文件是否存在：",
    OOT_ACCESS_LOG_PATH.exists()
)

display(oot_access_log)

OOT访问日志创建成功。
访问日志： D:\Risk_control project\metadata\oot_access_log.csv
文件是否存在： True


,access_id,access_time,project_stage,split_id,oot_week_range,purpose,data_scope,fields_accessed,feature_data_accessed,model_predictions_accessed,model_metrics_accessed,used_for_model_selection,final_oot_model_evaluation_opened,notes
0,OOT-BOUNDARY-001,2026-09-19 14:59:45,Step 3,split_v1,82-91,Define and freeze temporal split,Aggregated sample and event support only,case_id|date_decision|WEEK_NUM|target,False,False,False,False,False,"Only sample_count, event_count and event_rate ..."
